In [1]:
from pathlib import Path
import pandas as pd

In [2]:
processed = Path("../data/processed")
annual = pd.read_csv(processed / "faangm_annual_revenue.csv")
fin = pd.read_csv(processed / "faangm_financials.csv")
scores = pd.read_csv("../output/faangm_quality_scores.csv")

In [3]:
fin.pivot_table(index="ticker", columns="metric", values="value", aggfunc="count")

metric,capex,cfo,operating_income,rd,revenue
ticker,,,,,
AAPL,71.0,60.0,72.0,72.0,72.0
AMZN,70.0,72.0,72.0,NaN,72.0
GOOGL,46.0,46.0,47.0,47.0,47.0
META,58.0,58.0,60.0,60.0,60.0
MSFT,72.0,58.0,72.0,72.0,76.0
NFLX,72.0,72.0,72.0,72.0,72.0


### 기준 분기 (TTM)

In [4]:
latest_rev = fin[fin["metric"] == "revenue"].groupby("ticker")["quarter_end"].max()
used = scores.set_index("ticker")[["latest_quarter_end", "prior_year_quarter_end"]]
used.columns = ["ttm_end", "prior_ttm_end"]
used["latest_revenue_quarter"] = latest_rev
used

,ttm_end,prior_ttm_end,latest_revenue_quarter
ticker,,,
META,2026-06-30,2025-06-30,2026-06-30
MSFT,2026-06-30,2025-06-30,2026-06-30
NFLX,2026-06-30,2025-06-30,2026-06-30
AAPL,2026-06-27,2025-06-28,2026-06-27
GOOGL,2026-06-30,2025-06-30,2026-06-30
AMZN,2026-06-30,2025-06-30,2026-06-30


In [5]:
rev = annual.groupby("ticker")["revenue_billion"]

summary = annual.groupby("ticker").agg(latest_year=("fiscal_year", "last"), revenue=("revenue_billion", "last"))
summary["growth_pct"] = rev.apply(lambda s: s.pct_change().iloc[-1] * 100)
summary["cagr_5y"] = rev.apply(lambda s: ((s.iloc[-1] / s.iloc[-6]) ** (1 / 5) - 1) * 100)
summary = summary.sort_values("revenue", ascending=False).round(2)

summary.to_csv(processed / "faangm_revenue_summary.csv", encoding="utf-8-sig")
summary

,latest_year,revenue,growth_pct,cagr_5y
ticker,,,,
AMZN,2025,716.92,12.38,13.18
AAPL,2025,416.16,6.43,8.68
GOOGL,2025,402.84,15.09,17.15
MSFT,2026,331.84,17.79,14.57
META,2025,200.97,22.17,18.51
NFLX,2025,45.18,15.85,12.57


In [11]:
s = scores.set_index("company")
spend_bn = (s["capex"] + s["rd"].fillna(0)) / 1e9

metrics = s[["revenue_growth_yoy", "operating_margin", "fcf_margin"]].mul(100).round(2)
metrics["invest_bn"] = spend_bn.round(1)
metrics["growth_per_bn"] = (s["revenue_growth_yoy"] * 100 / spend_bn).round(2)
metrics

,revenue_growth_yoy,operating_margin,fcf_margin,invest_bn,growth_per_bn
company,,,,,
Meta,27.65,38.08,17.95,161.0,0.17
Microsoft,17.79,46.78,20.19,151.5,0.12
Netflix,16.02,29.68,23.06,4.5,3.54
Apple,14.24,33.17,29.28,52.9,0.27
Alphabet,20.05,33.11,11.95,201.4,0.10
Amazon,15.77,12.08,-1.50,173.0,0.09


In [12]:
score_cols = ["revenue_growth_score", "operating_margin_score", "fcf_margin_score", "investment_efficiency_score"]
ranking = scores.sort_values("quality_score", ascending=False).set_index("company")
ranking[["latest_quarter_end", "quality_score"] + score_cols].round(1)

,latest_quarter_end,quality_score,revenue_growth_score,operating_margin_score,fcf_margin_score,investment_efficiency_score
company,,,,,,
Meta,2026-06-30,76.7,100.0,83.3,50.0,66.7
Microsoft,2026-06-30,71.7,66.7,100.0,66.7,50.0
Netflix,2026-06-30,64.2,50.0,33.3,83.3,100.0
Apple,2026-06-27,63.3,16.7,66.7,100.0,83.3
Alphabet,2026-06-30,52.5,83.3,50.0,33.3,33.3
Amazon,2026-06-30,21.7,33.3,16.7,16.7,16.7


In [13]:
cases = {"base": [0.30, 0.25, 0.25, 0.20], "equal": [0.25] * 4, "no_invest": [0.40, 0.30, 0.30, 0.0]}
rank_check = pd.DataFrame({k: (ranking[score_cols] @ pd.Series(w, index=score_cols)).rank(ascending=False).astype(int)
                           for k, w in cases.items()})
rank_check

,base,equal,no_invest
company,,,
Meta,1,1,1
Microsoft,2,2,2
Netflix,3,4,5
Apple,4,3,4
Alphabet,5,5,3
Amazon,6,6,6
